# Neural Models Laboratory: Learning, Depth, Activations, and Output Layers
This notebook is a worked solution for the lab *Neural Models: Learning, Depth, Activations, and Output Layers* (XOR / redundant safety sensors).

**How to use it:** run *Kernel → Restart & Run All* (needs `torch`, `numpy`, `matplotlib`, `pandas`; CPU only, roughly 1–2 minutes in total). All numbers quoted in the write-up cells are produced by the code cells, so read the printed results and make sure the prose matches what *your* run shows.

**Structure**

| Task | Content |
|---|---|
| 1 | Problem specification and linear-separability argument |
| 2 | Model design and validation criteria |
| 3 | The LLM prompt and the first generated implementation |
| 4 | Execution and diagnosis (Parts A-D) |
| 5 | Three-class extension with softmax / cross-entropy |
| - | Result table, reflection questions, responsible use of LLMs |

## Task 1: Understand the problem before coding

**Input space:** $\mathcal X=\{(x_1,x_2)\mid x_1,x_2\in\{0,1\}\}$  **Output space:** $\mathcal Y=\{0,1\}$

**Labelled examples (XOR, "sensor disagreement"):**

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**Why one straight line cannot separate the classes.** Class 1 is the pair $(0,1),(1,0)$ and class 0 is the pair $(0,0),(1,1)$, i.e. the two diagonals of the unit square. The midpoint of the two class-1 points is $(0.5,0.5)$, which is also the midpoint of the two class-0 points. A half-plane is convex, so if it contains both class-1 points it contains $(0.5,0.5)$. But then it cannot exclude both class-0 points, because their midpoint would then lie outside it. Hence no single line works.

**Prediction for a single affine layer + sigmoid.** The model $p=\sigma(w_1x_1+w_2x_2+b)$ has exactly one linear decision boundary, so it cannot fit XOR. Because the labels are balanced and symmetric, the loss is minimised at $w=0,\,b=0$, i.e. $p=0.5$ for all four inputs, with binary cross-entropy $\ln 2\approx0.693$ and at most 2-3 of 4 labels right. (This is checked in Task 4, Part A.)

**Think about it: what scientific claim does XOR let us test with four points?** That the *kind* of representation matters more than the number of parameters. A linear (affine) model class provably cannot represent this function no matter how long we train or how many affine layers we stack, whereas a model with a nonlinear hidden layer can.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Xn = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
yn = np.array([0, 1, 1, 0])

print("Input space X:\n", Xn)
print("Output labels y:", yn)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for (x1, x2), lab in zip(Xn, yn):
    ax.scatter(x1, x2, c="tab:blue" if lab == 1 else "tab:orange", s=160, edgecolor="black", zorder=3)
    ax.text(x1 + 0.05, x2 + 0.05, f"y={lab}", fontsize=11)
ax.set_xlim(-0.2, 1.3); ax.set_ylim(-0.2, 1.3)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("XOR: blue = class 1, orange = class 0")
ax.grid(True, linestyle="--", alpha=0.4)
plt.show()

# Brute-force evidence: try many lines w1*x1 + w2*x2 + b > 0 and record the best accuracy
grid = np.linspace(-5, 5, 41)
W1g, W2g, Bg = np.meshgrid(grid, grid, grid, indexing="ij")
scores = W1g[..., None] * Xn[:, 0] + W2g[..., None] * Xn[:, 1] + Bg[..., None]   # (41,41,41,4)
acc = ((scores > 0).astype(int) == yn).mean(axis=-1)
print(f"Best accuracy over {acc.size} lines: {acc.max():.2f}  (a separating line would give 1.00)")

## Task 2: Design the intelligent agent

**Baseline design (on paper, before asking an LLM):**

- Architecture: 2 inputs → 2 hidden units → 1 output, i.e. $a^{(1)}=W^{(1)}x+b^{(1)},\;h=f(a^{(1)}),\;z=W^{(2)}h+b^{(2)},\;p=\sigma(z)$
- Hidden activation: sigmoid first (tanh and ReLU are compared in Part D)
- Output: sigmoid (implemented as a *logit* + `BCEWithLogitsLoss`, which is the numerically stable form)
- Loss: binary cross-entropy averaged over the four examples
- Optimiser: full-batch gradient descent (SGD, lr = 0.5, 5000 steps), random initialisation

**1. Why is the hidden nonlinearity scientifically necessary?**
Without it, $z=W^{(2)}(W^{(1)}x+b^{(1)})+b^{(2)}$ is again an affine function of $x$. Any stack of affine layers collapses into a single affine map, i.e. one straight decision boundary, which XOR does not allow. Depth alone adds nothing; the nonlinearity between layers is what lets the network build a new representation (two hidden features, roughly "OR-like" and "AND-like") in which the classes *are* linearly separable.

**2. Why sigmoid + binary cross-entropy?**
The target is one yes/no answer, so a sigmoid output gives a probability $p\in(0,1)$ and BCE is the negative log-likelihood of a Bernoulli variable. The pairing gives a clean logit gradient $\partial\mathcal L/\partial z=p-y$ that does not vanish when the prediction is confidently wrong (unlike sigmoid + squared error).

**3. Evidence that will count as successful learning (validation criteria):**

1. The loss falls clearly from its initial value (about 0.7) to a small value (e.g. below 0.1).
2. All four thresholded predictions equal the XOR labels, with probabilities near 0 or 1.
3. Gradients are nonzero at the start and during training (not stuck at zero or NaN).
4. Repeated runs: the outcome is reported over several random seeds, not one lucky seed.
5. Autograd's gradient agrees with an independent finite-difference (and hand-derived chain-rule) gradient.

**Think about it: the hidden units receive no target values, so what decides what each computes?**
Nothing tells the hidden units what to do. Backpropagation sends the output error $p-y$ backwards through $W^{(2)}$ and the activation derivative $f'(a^{(1)})$; that gives each hidden unit a gradient describing how its own output should change to reduce the final loss. The features the hidden units end up computing are therefore whatever is *useful for the output loss*, and random initialisation (asymmetry) decides which unit takes which role.

## Task 3: Use an LLM to generate a first implementation

**Prompt given to the LLM** (written *after* the specification in Task 2):

> Generate minimal PyTorch code for the following model and dataset. Do not change the architecture or task.
> Dataset (full batch): (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0.
> Model: 2–2–1 network, sigmoid hidden activation, one output logit trained with `BCEWithLogitsLoss` (sigmoid + binary cross-entropy).
> Use random weight initialisation, SGD with learning rate 0.5, full-batch training for 5000 steps on CPU.
> After training, report the final loss, all four probabilities, thresholded labels, and one parameter-gradient tensor.
> Set a random seed for reproducibility and explain each test in one sentence.

*(If you used different wording, replace this block with your exact prompt.)*

**Code inspection before running**

- **Forward pass:** `logits = model(X)`: $X\to\sigma(XW_1^\top+b_1)\to$ logit.
- **Scalar loss:** `loss = criterion(logits, y)`: mean BCE over the 4 examples.
- **Reverse-mode AD (backprop):** `loss.backward()` fills `.grad` for every parameter.
- **Parameter update:** `optimizer.step()`.
- **Verifiable without running:** architecture (2–2–1), targets and shapes `(4,1)`, that the criterion takes *logits* (no extra sigmoid in the model), the order `zero_grad → forward → loss → backward → step`, and the dataset values.
- **Needs execution and measurement:** whether the loss actually decreases, whether all four labels are right, gradient magnitudes, and sensitivity to the seed.

**Two changes made to the generated code** (before it is reused in Task 4): (1) the training loop is wrapped in a function that takes the seed and activation and returns the loss history, so the identical code can be reused for the later experiments; (2) the gradient of $W^{(1)}$ is recorded at step 0 (*early* gradient) as well as at the end, since Part D asks for an early-step norm.

**Think about it:** an LLM can produce code that runs but implements the wrong experiment, e.g. adding a second sigmoid before `BCEWithLogitsLoss`, silently changing the hidden size, or averaging the loss the wrong way. Those can be spotted by reading; whether learning succeeds cannot.

In [ ]:
import torch
import torch.nn as nn

# Reproducibility
torch.manual_seed(0)

# XOR dataset: four training examples
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32)
y = torch.tensor([0.0, 1.0, 1.0, 0.0], dtype=torch.float32).view(-1, 1)

# 2 -> 2 -> 1 network (output is a logit)
model = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 1))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

with torch.no_grad():
    print(f"Initial loss: {criterion(model(X), y).item():.6f}")

for epoch in range(5000):
    optimizer.zero_grad()
    logits = model(X)              # forward pass
    loss = criterion(logits, y)    # scalar loss
    loss.backward()                # reverse-mode autodiff
    optimizer.step()               # parameter update

with torch.no_grad():
    probs = torch.sigmoid(model(X))
    labels = (probs >= 0.5).float()
    print(f"Final loss:          {criterion(model(X), y).item():.6f}")
    print("Probabilities:      ", [round(v, 4) for v in probs.view(-1).tolist()])
    print("Thresholded labels: ", labels.view(-1).tolist())
    print("Target labels:      ", y.view(-1).tolist())
print("dL/dW1 (gradient from the last backward pass):")
print(model[0].weight.grad)

## Task 4: Execute, test, and diagnose the generated code

Shared setup used by all experiments below. Only justified engineering settings (seed, learning rate, steps) are ever changed; the task and architecture stay fixed. The gradient of $W^{(1)}$ is recorded at **step 0** (before any update) so that "early gradient norm" really means early.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32)
y = torch.tensor([0.0, 1.0, 1.0, 0.0], dtype=torch.float32).view(-1, 1)

LR, STEPS, N_SEEDS = 0.5, 5000, 10
bce = nn.BCEWithLogitsLoss()

ACTS = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh, "relu": nn.ReLU}

def build_model(act="sigmoid", seed=0, hidden=2, out=1):
    torch.manual_seed(seed)                       # seed -> random initialisation
    return nn.Sequential(nn.Linear(2, hidden), ACTS[act](), nn.Linear(hidden, out))

def train(model, X, y, criterion, lr=LR, steps=STEPS, snapshot_steps=()):
    # Full-batch SGD. Returns loss history, W1-gradient at step 0, and optional W1 snapshots.
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    losses, snaps, early_grad = [], {}, None
    for step in range(steps):
        opt.zero_grad()
        loss = criterion(model(X), y)             # forward + scalar loss
        loss.backward()                           # backprop
        if step == 0:
            early_grad = model[0].weight.grad.clone()
        losses.append(loss.item())
        opt.step()                                # parameter update
        if step in snapshot_steps:
            snaps[step] = model[0].weight.detach().clone()
    return {"losses": losses, "early_grad": early_grad, "snaps": snaps}

def evaluate(model, X, y, criterion=bce):
    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        return {"loss": criterion(logits, y).item(), "probs": probs.view(-1),
                "preds": preds.view(-1), "correct": bool((preds == y).all())}

def run(act, seed, **kw):
    m = build_model(act, seed)
    tr = train(m, X, y, bce, **kw)
    return m, tr, evaluate(m, X, y)

print("Setup ready. torch", torch.__version__)

### Part A: Basic learning check

**A0. Check the Task 1 prediction:** a single affine layer + sigmoid should stay near $p=0.5$, loss $\approx\ln 2=0.693$.

In [ ]:
torch.manual_seed(0)
linear = nn.Sequential(nn.Linear(2, 1))          # one affine map -> logit -> sigmoid (inside the loss)
train(linear, X, y, bce)
ev = evaluate(linear, X, y)
print(f"Linear model final loss: {ev['loss']:.4f}   (ln 2 = {np.log(2):.4f})")
print("Probabilities:", [round(v, 3) for v in ev["probs"].tolist()])
print("Predictions  :", ev["preds"].tolist(), " targets:", y.view(-1).tolist())
print("All four correct?", ev["correct"])

**A1. The 2–2–1 sigmoid network.** A tiny 2–2–1 network can get stuck for some initialisations, so besides the main run we test several seeds (validation criterion 4). If seed 0 fails we switch to the first seed that works; this is a justified engineering change (seed only).

In [ ]:
scan = []
for s in range(N_SEEDS):
    _, _, ev = run("sigmoid", s)
    scan.append((s, ev["loss"], ev["correct"]))
solved = [s for s, _, ok in scan if ok]
print("seed  final loss  4/4 correct")
for s, l, ok in scan:
    print(f"{s:4d}  {l:10.4f}  {ok}")
print(f"\nSeeds that solved XOR: {len(solved)}/{N_SEEDS}")

main_seed = 0 if (0 in solved or not solved) else solved[0]
print("Main seed used for later parts:", main_seed)

model = build_model("sigmoid", main_seed)
init_loss = evaluate(model, X, y)["loss"]
tr = train(model, X, y, bce)
ev = evaluate(model, X, y)

print(f"\nInitial loss: {init_loss:.4f}")
print(f"Final loss:   {ev['loss']:.4f}")
print("Final probabilities:", [round(v, 4) for v in ev["probs"].tolist()])
print("Thresholded labels :", ev["preds"].tolist())
print("Target labels      :", y.view(-1).tolist())
print("All four labels correct?", ev["correct"])

plt.figure(figsize=(5.5, 3.5))
plt.semilogy(tr["losses"])
plt.xlabel("step"); plt.ylabel("BCE loss"); plt.title(f"Sigmoid 2-2-1, seed {main_seed}")
plt.grid(True, alpha=0.3); plt.show()

**Reading the result.** A successful run shows (i) a loss that drops from about 0.7 to a small value, usually after a plateau near 0.69 where the network is still "linear-like", (ii) probabilities near 0/1, and (iii) all four labels correct. A run that ends at loss ≈ 0.69 or with only 2-3 correct labels is a local-minimum / plateau failure of this tiny network, not evidence against the design; the seed scan quantifies how often it happens.

### Part B: Backpropagation check

`parameter.grad` after `loss.backward()` holds $\partial\mathcal L/\partial W^{(1)}$, the derivative of the *scalar* loss with respect to each entry of the first-layer weight matrix (same shape as `weight`, here $2\times2$). Because the loss is the *mean* over the 4 examples, $\mathcal L=\tfrac14\sum_i\mathcal L_i$, so by linearity of the derivative $\partial\mathcal L/\partial W^{(1)}=\tfrac14\sum_i\partial\mathcal L_i/\partial W^{(1)}$: the gradient is the average of the example-wise gradients.

We check the autograd gradient three independent ways (float64): per-example average, hand-written chain rule, and central finite differences.

In [ ]:
m = build_model("sigmoid", main_seed).double()
Xd, yd = X.double(), y.double()

# 1) autograd on the mean loss
m.zero_grad(); bce(m(Xd), yd).backward()
g_auto = m[0].weight.grad.clone()
print("autograd dL/dW1:\n", g_auto)

# 2) mean of the four per-example gradients
per_example = []
for i in range(4):
    m.zero_grad(); bce(m(Xd[i:i+1]), yd[i:i+1]).backward()
    per_example.append(m[0].weight.grad.clone())
g_mean = torch.stack(per_example).mean(0)
print("\nmax |mean(per-example grads) - autograd| =", (g_mean - g_auto).abs().max().item())

# 3) hand-written chain rule:  dL/dz = (p - y)/N ; dh = dz W2 ; da = dh * h(1-h) ; dW1 = da^T X
with torch.no_grad():
    a1 = m[0](Xd); h = torch.sigmoid(a1); z = m[2](h); p = torch.sigmoid(z)
    dz = (p - yd) / 4
    dh = dz @ m[2].weight
    da = dh * h * (1 - h)
    g_manual = da.T @ Xd
print("max |hand-derived chain rule - autograd| =", (g_manual - g_auto).abs().max().item())

# 4) central finite differences on every entry of W1
eps, g_fd = 1e-6, torch.zeros_like(g_auto)
W = m[0].weight
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        with torch.no_grad():
            W[i, j] += eps;   lp = bce(m(Xd), yd).item()
            W[i, j] -= 2*eps; lm = bce(m(Xd), yd).item()
            W[i, j] += eps
        g_fd[i, j] = (lp - lm) / (2 * eps)
print("finite-difference dL/dW1:\n", g_fd)
print("max |finite difference - autograd| =", (g_fd - g_auto).abs().max().item())

All differences should be on the order of $10^{-9}$ or smaller, so autograd is computing exactly $\partial\mathcal L/\partial W^{(1)}$. Note the chain: the output error $(p-y)/N$ is multiplied by $W^{(2)}$ (sending it to the hidden layer), then by the local derivative $h(1-h)$ (through the sigmoid), then by the inputs $x$ (to get the weight gradient). That is backpropagation.

### Part C: Symmetry experiment

Same architecture, but every parameter is initialised to a constant. We inspect the two rows of $W^{(1)}$ (one row per hidden unit) over training.

- **All zeros:** shows what happens in the extreme case.
- **All 0.5:** a nonzero constant, so learning does happen, but symmetry is still unbroken. This isolates the symmetry effect.

In [ ]:
def const_init(model, v):
    with torch.no_grad():
        for p in model.parameters():
            p.fill_(v)

snap_steps = (0, 1, 10, 100, 1000, STEPS - 1)
for v in (0.0, 0.5):
    m = build_model("sigmoid", 0)
    const_init(m, v)
    tr = train(m, X, y, bce, snapshot_steps=snap_steps)
    ev = evaluate(m, X, y)
    print(f"===== all parameters initialised to {v} =====")
    print(f"early gradient norm ||dL/dW1||_2 at step 0: {tr['early_grad'].norm().item():.3e}")
    for st, w in tr["snaps"].items():
        diff = (w[0] - w[1]).abs().max().item()
        print(f"step {st:5d}  row0={w[0].numpy().round(4)}  row1={w[1].numpy().round(4)}  max|row0-row1|={diff:.1e}")
    print(f"final loss {ev['loss']:.4f}, probs {[round(p, 3) for p in ev['probs'].tolist()]}, 4/4 correct: {ev['correct']}\n")

# Contrast: random initialisation breaks the symmetry
m = build_model("sigmoid", main_seed)
tr = train(m, X, y, bce, snapshot_steps=(0, STEPS - 1))
print("random init, rows after training:\n", m[0].weight.detach().numpy().round(3))

**Explanation.**

- With **all-zero** weights every gradient is exactly zero for this dataset (the hidden activations are all 0.5, $p=0.5$, the labels are balanced, and $W^{(2)}=0$ blocks the error from reaching $W^{(1)}$), so nothing moves at all: the loss stays at $\ln2$ and the outputs stay at 0.5.
- With **all-0.5** weights the network does update, but the two rows of $W^{(1)}$ remain identical at every step (max difference 0). Both hidden units receive the same inputs and have the same weights, so they compute the same activation; they also have the same outgoing weight, so they receive exactly the same backpropagated gradient and get exactly the same update. Identical in → identical out → identical gradient → identical update, by induction forever.
- Two identical hidden units are effectively **one** unit. The network then computes a function of $x_1+x_2$ only, monotone in it, while XOR needs $f(0)<0.5$, $f(1)>0.5$, $f(2)<0.5$. So it can never get all four labels right.
- Random initialisation **breaks the symmetry**, so the units can specialise, which is why the random-init rows above are different.

### Part D: Activation experiment

Only the hidden activation changes (sigmoid / tanh / ReLU); seed, learning rate, steps and data are identical. Per activation we record the final loss, whether all four labels are right, the norm of $\partial\mathcal L/\partial W^{(1)}$ at **step 0**, and, because one seed is anecdotal, the number of seeds solved out of `N_SEEDS`.

In [ ]:
rows, curves, trained = [], {}, {}
for act in ("sigmoid", "tanh", "relu"):
    m, tr, ev = run(act, main_seed)
    trained[act] = m
    curves[act] = tr["losses"]
    wins = sum(run(act, s)[2]["correct"] for s in range(N_SEEDS))
    rows.append({
        "Hidden activation": act,
        "Final loss": round(ev["loss"], 5),
        "4/4 correct?": "Yes" if ev["correct"] else "No",
        "Early ||dL/dW1||_2 (step 0)": float(f"{tr['early_grad'].norm().item():.4g}"),
        f"Seeds solved (of {N_SEEDS})": wins,
    })

result_table = pd.DataFrame(rows)
display(result_table)

plt.figure(figsize=(6, 3.6))
for act, c in curves.items():
    plt.semilogy(c, label=act)
plt.xlabel("step"); plt.ylabel("BCE loss"); plt.legend(); plt.grid(True, alpha=0.3)
plt.title(f"Loss curves by hidden activation (seed {main_seed})"); plt.show()

**Saturated sigmoid vs. dead ReLU: how to tell them apart.** Both give tiny gradients but by different mechanisms, and you can tell which by inspecting pre-activations $a=W^{(1)}x+b^{(1)}$ and activations $h=f(a)$:

- **Saturated sigmoid/tanh:** $|a|$ is large, $h$ is pinned near 0 or 1 (or ±1), and the derivative $h(1-h)$ (or $1-h^2$) is small **but not exactly zero**. The gradient is tiny and can in principle still recover.
- **Dead/inactive ReLU:** $a\le0$ for the inputs, $h=0$ exactly, and the derivative is **exactly 0**. If a unit has $a\le0$ for *all four* inputs it receives no gradient at all and can never recover.

In [ ]:
def diagnose(model, act):
    with torch.no_grad():
        a = model[0](X)               # pre-activations, shape (4, 2 hidden units)
        h = model[1](a)               # activations
    print(f"--- {act} ---")
    print("pre-activations a (rows = the 4 inputs, cols = hidden units):\n", a.numpy().round(2))
    if act == "relu":
        active_any = (a > 0).any(dim=0)
        print("fraction of (input, unit) pairs with a<=0 (derivative exactly 0):", round((a <= 0).float().mean().item(), 2))
        print("hidden unit active on at least one input?", active_any.tolist(), "(False = dead unit)")
    else:
        d = h * (1 - h) if act == "sigmoid" else 1 - h ** 2
        print("mean |a|:", round(a.abs().mean().item(), 2), "| mean derivative f'(a):", round(d.mean().item(), 4), "| min f'(a):", round(d.min().item(), 6))

for act, m in trained.items():
    diagnose(m, act)

**Interpretation of the table (read together with your printed results).**

- **Scientific explanation (why activations differ):** the gradient reaching $W^{(1)}$ contains the factor $f'(a)$. For sigmoid $f'\le0.25$ and it shrinks further when the unit saturates, so gradients are small and learning is slow, typically with a long plateau. For tanh $f'\le1$ and outputs are zero-centred, so the early gradient is larger and learning is faster. For ReLU, $f'=1$ for active units and exactly $0$ for inactive ones; with only two hidden units, one unit that is inactive on all inputs is *dead*, the network then has effectively one hidden feature and cannot represent XOR.
- **Engineering observation (what we measured):** the final loss, the 4/4 flag, the step-0 gradient norm and the seed success rate in the table. The early-gradient norm is a diagnostic, not a success criterion: a small value means slow start, not failure, and a large value does not guarantee success.
- Four data points and 2 hidden units cannot establish that any activation is universally best; the table describes *this* experiment (this size, learning rate, step count and these seeds). Update the sentences above if your numbers show something different.

## Task 5: Extend the task, three-class decision

Classes: 0 = both inactive $(0,0)$; 1 = disagree $(0,1),(1,0)$; 2 = both active $(1,1)$. Only the output layer and loss change: 2 inputs → 2 hidden units (sigmoid) → **3 logits**, trained with multiclass cross-entropy (`CrossEntropyLoss` = log-softmax + negative log-likelihood, applied to raw logits).

**Predictions before running:**

1. **Final weight matrix shape:** PyTorch stores `Linear(2, 3).weight` as `(out, in) = (3, 2)`, i.e. one row of hidden-unit weights per class. This is the same as a $2\times3$ hidden-to-class matrix in the $h^\top W$ convention.
2. **Logits per example:** 3, one per class.
3. **Softmax sums to one:** $p_k=e^{z_k}/\sum_j e^{z_j}$, so $\sum_kp_k=\sum_ke^{z_k}/\sum_je^{z_j}=1$ by construction.
4. **Why the logit gradient is $p-y$:** with $\mathcal L=-\sum_ky_k\log p_k=-z_c+\log\sum_je^{z_j}$ for true class $c$, $\partial\mathcal L/\partial z_k=-y_k+p_k$. With the mean over $N=4$ examples this becomes $(p-y)/4$ per example.

In [ ]:
y3 = torch.tensor([0, 1, 1, 2], dtype=torch.long)     # 0:(0,0)  1:(0,1),(1,0)  2:(1,1)
ce = nn.CrossEntropyLoss()

def build_cls(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 3))   # only the output size changed

def run_cls(seed):
    m = build_cls(seed)
    tr = train(m, X, y3, ce)
    with torch.no_grad():
        logits = m(X); probs = torch.softmax(logits, dim=1)
        ok = bool((probs.argmax(1) == y3).all())
        return m, tr, ce(logits, y3).item(), ok

results = [(s,) + run_cls(s)[2:] for s in range(N_SEEDS)]
solved3 = [s for s, _, ok in results if ok]
print(f"Seeds solving the 3-class task: {len(solved3)}/{N_SEEDS}")
seed3 = 0 if (0 in solved3 or not solved3) else solved3[0]
model3, tr3, final_loss3, ok3 = run_cls(seed3)

with torch.no_grad():
    logits = model3(X); probs = torch.softmax(logits, dim=1)

print(f"\nseed used: {seed3} | final loss: {final_loss3:.5f} | all 4 correct: {ok3}")
print("Final weight matrix shape (PyTorch out x in):", tuple(model3[2].weight.shape))
print("Logits per example:", logits.shape[1])
print("\nClass probabilities for the four inputs (columns = class 0,1,2):\n", probs.numpy().round(4))
print("Predicted classes:", probs.argmax(1).tolist(), "| targets:", y3.tolist())

ex = 1                                  # inspect one example, (0,1)
v = probs[ex]
print(f"\nSoftmax vector for example {ex}:", v.numpy().round(6), "| sum =", v.sum().item())

In [ ]:
# (i) Verify the logit gradient is (p - y)/N numerically
z = model3(X).detach().requires_grad_(True)
ce(z, y3).backward()
onehot = F.one_hot(y3, num_classes=3).float()
expected = (torch.softmax(z.detach(), dim=1) - onehot) / X.shape[0]
print("max |autograd dL/dz - (p - y)/N| =", (z.grad - expected).abs().max().item())
print("Example", ex, "-> p - y =", (torch.softmax(z.detach(), dim=1)[ex] - onehot[ex]).numpy().round(5))

# (ii) Optional diagnostic: add the same constant to all logits
zz = logits[ex]
print("\nsoftmax(z)        :", torch.softmax(zz, 0).numpy().round(6))
print("softmax(z + 100)  :", torch.softmax(zz + 100, 0).numpy().round(6))
print("difference norm   :", (torch.softmax(zz + 100, 0) - torch.softmax(zz, 0)).norm().item())

# (iii) Why implementations subtract the max logit
naive = torch.exp(zz + 100) / torch.exp(zz + 100).sum()
print("\nnaive exp(z+100)/sum (float32):", naive.numpy(), "<- overflow to inf gives inf/inf = nan")
shifted = zz + 100
stable = torch.exp(shifted - shifted.max())
print("subtract max first             :", (stable / stable.sum()).numpy().round(6))

**Explanation.**

- **Shift invariance:** $\dfrac{e^{z_k+c}}{\sum_je^{z_j+c}}=\dfrac{e^{c}e^{z_k}}{e^{c}\sum_je^{z_j}}=p_k$, so a common constant cancels. Stable implementations exploit this by subtracting $\max_jz_j$, which makes the largest exponent $e^0=1$ and prevents overflow (and the inf/inf = NaN seen above) without changing the probabilities.
- **The $p-y$ gradient** is just "predicted probability minus target": zero when the model is right and confident, and largest when it puts probability on the wrong class. The same form arises for sigmoid + BCE (Task 2), which is why output layer and loss should be chosen together.
- **Side remark:** the three classes are ordered along $x_1+x_2\in\{0,1,2\}$, so a purely linear softmax could already separate them. This variant is *not* XOR-hard, unlike the binary problem.

**Think about it: next-token prediction as classification over a huge vocabulary.**
*Stays the same:* logits → softmax → cross-entropy, the $p-y$ gradient, shift invariance and the max-subtraction trick, and backpropagation through the last layer $\partial\mathcal L/\partial W_{\text{out}}=(p-y)h^\top$.
*Changes dramatically:* the number of classes goes from 3 to tens of thousands, so the output matrix and the softmax normalisation become a major cost (leading to tricks such as tied embeddings, sampled/hierarchical softmax, fused kernels); the hidden representation $h$ is no longer two sigmoid units of a 2-d input but the output of a deep network (embeddings + many transformer layers) over a long context; the data are sequences with autoregressive targets, not four fixed points; and training uses minibatches, adaptive optimisers and huge datasets.

## Result table (activation experiment)

The table is generated by the Part D code above (Restart & Run All to refresh); it is displayed again here for the report. The "early" gradient norm is measured at **step 0**, before any parameter update.

In [ ]:
display(result_table)

## Reflection questions

**1. What did the XOR experiment demonstrate about depth vs. nonlinearity?**
Depth by itself does nothing: stacked affine layers collapse to one affine map, and the single-layer baseline sat at $p=0.5$, loss $\ln2$. Adding a hidden layer with a nonlinear activation is what made XOR solvable. Nonlinearity between layers, not the layer count, provides the extra representational power.

**2. In the successful run, what evidence showed that backpropagation supplied a useful learning signal, rather than merely a nonzero gradient?**
The loss actually fell from about 0.7 to a small value, the probabilities moved to 0/1 and all four labels became correct, and autograd matched the finite-difference and hand-derived chain-rule gradients (so the gradient was the *correct* one). By contrast, the zero-initialised run had exactly zero gradient, and the symmetric 0.5 run had nonzero gradients that never gave all four labels correct. A nonzero gradient is necessary but not sufficient.

**3. Why did identical/zero initialisation prevent the two hidden units from learning distinct features?**
Identical hidden units compute the same output and, since they have the same outgoing weight, receive the same backpropagated error and hence the same update, so they stay identical for all steps (the rows of $W^{(1)}$ remained equal). With all-zero weights the gradients are in fact exactly zero. Random initialisation breaks the symmetry.

**4. How did changing the hidden activation affect the observed gradient? Distinguish scientific explanation from engineering observation.**
*Scientific:* the gradient to $W^{(1)}$ contains $f'(a)$: at most 0.25 (and near 0 when saturated) for sigmoid, up to 1 for tanh, and exactly 1 or 0 for ReLU (0 = inactive/dead). *Engineering:* what we observed was the step-0 gradient norm, final loss, success flag and seed success rate in the table, for this seed and setup only. These numbers illustrate the mechanism; they do not prove one activation is universally better.

**5. Why must the output layer and loss be selected together?**
The output layer defines what the numbers mean (a Bernoulli probability for sigmoid, a categorical distribution for softmax) and the loss must be the matching likelihood. Only then does the logit gradient have the clean form $p-y$ and stay well-behaved, and only then are the outputs interpretable as probabilities. Mismatches (e.g. softmax with squared error, or a sigmoid followed by a loss that expects logits) give weak or wrong gradients and misleading outputs.

**6. One example where the LLM improved productivity, and one where human verification was essential.**
*Productivity:* it produced the boilerplate PyTorch training loop and the finite-difference gradient check quickly, so more time went on interpreting results. *Verification:* checking that the loss was applied to logits (not to sigmoid outputs), that the "early" gradient was captured at step 0 rather than after training, and that the reported claims (e.g. the symmetry outcome and the activation comparison) matched the *measured* output rather than what a plausible-sounding explanation would say.

**7. Which tests would you keep if the model were scaled up, and which would become too expensive?**
*Keep:* watching the loss curve, overfitting a tiny batch as a sanity check, checking predictions on a small validation set, monitoring gradient norms (NaN/zero/exploding), testing several seeds where affordable, and checking shapes and the loss/output pairing. *Too expensive:* exhaustive finite-difference gradient checks over all parameters (cost grows with the number of parameters, each needing extra forward passes); use them only on a tiny model or a few randomly chosen entries. Running many seeds of a large model is also costly.

## Responsible use of LLMs (verification log)

- The problem specification (Task 1) and the model design and validation criteria (Task 2) were written **before** the LLM was asked for code.
- The generated code was inspected line by line for the forward pass, loss, `backward()` and optimiser step (Task 3) before being run.
- Every claim in this notebook is backed by a printed measurement (loss, predictions, gradient checks, symmetry rows, activation table, softmax checks). The LLM is a collaborator for engineering; the scientific reasoning and its interpretation stay the author's responsibility.